# Sequence model (Week 5)

Predict **EOL** from the first **100 cycles** of per-cycle data in `cycle_summary.csv`.

| Step | What |
|------|------|
| A | Build inputs — `(134, 100, 4)` tensor + labels |
| B | Scale, GRU, training loop (fixed settings) |
| C | Tune on val → test metrics + figure |

Each battery: **100 cycles × 4 signals** — SOH, resistance, energy efficiency, temperature. Same split as Week 4 (`cell_split.csv`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
TARGETS_PATH = ROOT / 'data' / 'cell_targets.csv'
SUMMARY_PATH = ROOT / 'data' / 'processed' / 'cycle_summary.csv'
SPLIT_PATH = ROOT / 'data' / 'processed' / 'cell_split.csv'

SEQ_LEN = 100
CYCLE_MIN = 1  # exclude formation cycle 0
CYCLE_MAX = SEQ_LEN
SEQUENCE_COLS = (
    'soh',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
)
TARGET = 'EOL'

print('Project root:', ROOT)
print('Sequence length:', SEQ_LEN, 'cycles')
print('Channels:', ', '.join(SEQUENCE_COLS))

In [ ]:
targets = pd.read_csv(TARGETS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

labels = targets.merge(split_df, on=['file_id', 'cell_id'], validate='one_to_one')

assert len(targets) == 134
assert len(labels) == 134
assert labels['file_id'].is_unique
assert set(labels['split']) == {'train', 'val', 'test'}

print(f'Cells: {len(labels)}')
print(labels['split'].value_counts().sort_index().to_string())
print(f'EOL range: {labels[TARGET].min()} – {labels[TARGET].max()} cycles')

In [ ]:
summary_cols = [
    'file_id',
    'cell_id',
    'cycle_index',
    'discharge_capacity',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
]
cycle_summary = pd.read_csv(SUMMARY_PATH, usecols=summary_cols)
cycle_summary = cycle_summary[
    (cycle_summary['cycle_index'] >= CYCLE_MIN)
    & (cycle_summary['cycle_index'] <= CYCLE_MAX)
].copy()

print(f'cycle_summary rows (cycles {CYCLE_MIN}–{CYCLE_MAX}): {len(cycle_summary):,}')
print(f'Unique cells in summary: {cycle_summary["file_id"].nunique()}')

In [ ]:
def build_sequence(group: pd.DataFrame, initial_capacity: float) -> np.ndarray:
    """Return array shape (SEQ_LEN, n_channels) for one cell."""
    g = group.sort_values('cycle_index')
    expected_cycles = np.arange(CYCLE_MIN, CYCLE_MAX + 1)
    if not np.array_equal(g['cycle_index'].to_numpy(), expected_cycles):
        missing = set(expected_cycles) - set(g['cycle_index'])
        raise ValueError(f'Missing cycles {sorted(missing)[:5]}... (need {SEQ_LEN} consecutive cycles)')

    soh = g['discharge_capacity'].to_numpy(dtype=float) / initial_capacity
    out = np.column_stack([
        soh,
        g['dc_internal_resistance'].to_numpy(dtype=float),
        g['energy_efficiency'].to_numpy(dtype=float),
        g['temperature_average'].to_numpy(dtype=float),
    ])
    return out


sequences = []
y = []
meta_rows = []

for row in labels.itertuples(index=False):
    group = cycle_summary[cycle_summary['file_id'] == row.file_id]
    seq = build_sequence(group, row.initial_capacity)
    sequences.append(seq)
    y.append(row.EOL)
    meta_rows.append({
        'file_id': row.file_id,
        'cell_id': row.cell_id,
        'split': row.split,
        TARGET: row.EOL,
        'initial_capacity': row.initial_capacity,
    })

X = np.stack(sequences, axis=0)
y = np.array(y, dtype=float)
meta = pd.DataFrame(meta_rows)

print('X shape:', X.shape, '  (cells, cycles, channels)')
print('y shape:', y.shape)
print('Channels:', list(SEQUENCE_COLS))

In [ ]:
assert X.shape == (134, SEQ_LEN, len(SEQUENCE_COLS))
assert len(y) == 134
assert not np.isnan(X).any(), 'NaN in sequence tensor — check cycle_summary'
assert (X[:, :, 0] > 0).all(), 'SOH should be positive'

for split_name in ('train', 'val', 'test'):
    n = (meta['split'] == split_name).sum()
    assert n in (94, 20), f'unexpected {split_name} count: {n}'

print('Checks passed.')
print()
print('Example — first train cell, cycle 1 and cycle 100:')
train_idx = meta.index[meta['split'] == 'train'][0]
print(meta.loc[train_idx, ['file_id', TARGET, 'split']].to_string())
print('cycle 1:', dict(zip(SEQUENCE_COLS, X[train_idx, 0].round(4))))
print(f'cycle {SEQ_LEN}:', dict(zip(SEQUENCE_COLS, X[train_idx, -1].round(4))))

## Step B — scale, GRU, train

1. **Scale inputs** — each channel, fit on train only (`StandardScaler`).
2. **Scale EOL target** — fit mean/std on **train** EOL only; train on scaled values; convert back for MAE. (Raw EOL is ~200–2200; without this the GRU learns too slowly.)
3. **GRU** reads cycles 1→100 → predicts EOL.
4. **Train** on 94 cells; watch **val MAE**; early stop.

Fixed settings for now (Step C tunes on val):

In [ ]:
import json

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_CHANNELS = len(SEQUENCE_COLS)

# Defaults for Step B demo; Step C grid-search overrides these
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.2
LEARNING_RATE = 1e-3
BATCH_SIZE = 16
MAX_EPOCHS = 200
PATIENCE = 20

METRICS_PATH = ROOT / 'results' / 'metrics' / 'gru_sequence.json'
FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_sequence.png'

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_mask = meta['split'].eq('train').to_numpy()
val_mask = meta['split'].eq('val').to_numpy()
test_mask = meta['split'].eq('test').to_numpy()

scaler_x = StandardScaler()
n_cells, seq_len, n_feat = X.shape
scaler_x.fit(X[train_mask].reshape(-1, n_feat))
X_scaled = scaler_x.transform(X.reshape(-1, n_feat)).reshape(X.shape)

y_mean = y[train_mask].mean()
y_std = y[train_mask].std()
y_scaled = (y - y_mean) / y_std

X_train = X_scaled[train_mask]
X_val = X_scaled[val_mask]
X_test = X_scaled[test_mask]
y_train = y_scaled[train_mask]
y_val = y_scaled[val_mask]
y_test = y_scaled[test_mask]
y_train_raw = y[train_mask]
y_val_raw = y[val_mask]
y_test_raw = y[test_mask]

print(f'train {len(y_train)} | val {len(y_val)} | test {len(y_test)}')
print(f'EOL train mean {y_mean:.0f}, std {y_std:.0f}')

In [ ]:
def unscale_predictions(pred_scaled: np.ndarray) -> np.ndarray:
    return pred_scaled * y_std + y_mean


class GRUEOLRegressor(nn.Module):
    """Read (batch, cycles, channels) → predict scaled EOL per cell."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        last = self.dropout(out[:, -1, :])
        return self.head(last).squeeze(-1)


def make_loader(x_arr: np.ndarray, y_arr: np.ndarray, shuffle: bool) -> DataLoader:
    ds = TensorDataset(
        torch.tensor(x_arr, dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


def predict_cycles(model: nn.Module, x_arr: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.tensor(x_arr, dtype=torch.float32).to(device)).cpu().numpy()
    return unscale_predictions(pred_scaled)


def eval_mae_cycles(model: nn.Module, x_arr: np.ndarray, y_raw: np.ndarray) -> float:
    pred = predict_cycles(model, x_arr)
    return float(np.abs(pred - y_raw).mean())


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def train_gru(
    hidden_size: int,
    num_layers: int,
    dropout: float,
    learning_rate: float,
    verbose: bool = False,
) -> tuple[GRUEOLRegressor, float, dict[str, torch.Tensor]]:
    torch.manual_seed(RANDOM_STATE)
    model = GRUEOLRegressor(N_CHANNELS, hidden_size, num_layers, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()
    train_loader = make_loader(X_train, y_train, shuffle=True)

    best_val_mae = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()

        val_mae = eval_mae_cycles(model, X_val, y_val_raw)
        if val_mae < best_val_mae - 1e-4:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch == 1 or epoch % 20 == 0):
            train_mae = eval_mae_cycles(model, X_train, y_train_raw)
            print(f'  epoch {epoch:3d}  train {train_mae:6.1f}  val {val_mae:6.1f}')

        if epochs_no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_val_mae, best_state

In [ ]:
# Step B — quick run with default settings
model, best_val_mae, _ = train_gru(HIDDEN_SIZE, NUM_LAYERS, DROPOUT, LEARNING_RATE, verbose=True)
print(f'\nDefault settings — best val MAE: {best_val_mae:.1f} cycles')

## Step C — tune on val, score test

Grid search on **validation MAE** (same idea as Week 4 RF/XGBoost tuning).  
**Test set** is scored once at the end for final metrics + figure.

Compare to Week 4 XGBoost test MAE **~85 cycles**.

In [ ]:
param_grid = [
    {
        'hidden_size': h,
        'num_layers': l,
        'dropout': d,
        'learning_rate': lr,
    }
    for h in (32, 64)
    for l in (1, 2)
    for d in (0.1, 0.2)
    for lr in (1e-3, 3e-4)
]

results = []
best_model = None
best_params = None
best_val_mae = float('inf')
best_state = None

for i, params in enumerate(param_grid, start=1):
    print(f'[{i}/{len(param_grid)}] {params}')
    model, val_mae, state = train_gru(**params)
    row = {**params, 'val_mae': val_mae}
    results.append(row)
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_params = params
        best_model = model
        best_state = state
    print(f'  -> val MAE {val_mae:.1f}')

print('\nBest params:', best_params)
print(f'Best val MAE: {best_val_mae:.1f} cycles')
pd.DataFrame(results).sort_values('val_mae').head(8).round(2)

In [ ]:
best_model.load_state_dict(best_state)

y_pred_train = predict_cycles(best_model, X_train)
y_pred_val = predict_cycles(best_model, X_val)
y_pred_test = predict_cycles(best_model, X_test)

metrics = {
    'model': 'gru_sequence',
    'sequence_len': SEQ_LEN,
    'channels': list(SEQUENCE_COLS),
    'best_params': best_params,
    'split': {
        'train': len(y_train),
        'val': len(y_val),
        'test': len(y_test),
        'random_state': RANDOM_STATE,
    },
    'train': regression_metrics(y_train_raw, y_pred_train),
    'val': regression_metrics(y_val_raw, y_pred_val),
    'test': regression_metrics(y_test_raw, y_pred_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE': [metrics[s]['mae'] for s in ('train', 'val', 'test')],
    'RMSE': [metrics[s]['rmse'] for s in ('train', 'val', 'test')],
    'MAPE (%)': [metrics[s]['mape'] for s in ('train', 'val', 'test')],
}).round(2)

In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(metrics, indent=2))
print('Saved metrics to', METRICS_PATH)

test_mae = metrics['test']['mae']
print(f'\nGRU test MAE: {test_mae:.1f} cycles')
print('XGBoost test MAE (Week 4): ~85 cycles')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_test_raw.min(), y_pred_test.min())
hi = max(y_test_raw.max(), y_pred_test.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.scatter(y_test_raw, y_pred_test, alpha=0.85, edgecolors='white', linewidths=0.5)
ax.set_xlabel('True EOL (cycles)')
ax.set_ylabel('Predicted EOL (cycles)')
ax.set_title(f'GRU sequence model — test set (n={len(y_test_raw)}, MAE={test_mae:.0f} cycles)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper left')
fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', FIGURE_PATH)